In [2]:
import pandas as pd
df = pd.read_csv("data/Airlines.csv")
print(df.shape)
print(df.head())
print(df.dtypes)
print(df.isnull().sum())

(539383, 9)
   id Airline  Flight AirportFrom AirportTo  DayOfWeek  Time  Length  Delay
0   1      CO     269         SFO       IAH          3    15     205      1
1   2      US    1558         PHX       CLT          3    15     222      1
2   3      AA    2400         LAX       DFW          3    20     165      1
3   4      AA    2466         SFO       DFW          3    20     195      1
4   5      AS     108         ANC       SEA          3    30     202      0
id             int64
Airline          str
Flight         int64
AirportFrom      str
AirportTo        str
DayOfWeek      int64
Time           int64
Length         int64
Delay          int64
dtype: object
id             0
Airline        0
Flight         0
AirportFrom    0
AirportTo      0
DayOfWeek      0
Time           0
Length         0
Delay          0
dtype: int64


In [3]:
for col in ["Airline", "AirportFrom", "AirportTo"]:
    print(col, df[col].nunique())

Airline 18
AirportFrom 293
AirportTo 293


In [ ]:
"""
use lightgbm in the case that we consider the environment is stationary. i want to visulize the performance of lightgbm in the stationary environment. 
"""

In [ ]:
"use model\\LightGBM.py to train a lightgbm model and evaluate its performance on the test set. visualize the results using matplotlib."
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
import pandas as pd

import matplotlib.pyplot as plt

# choose a target column from common airline delay labels
target_candidates = ["ArrDelay", "Delay", "IsDelayed", "Cancelled", "FlightStatus"]
if 'df' not in globals():
    df = pd.read_csv("data/Airlines.csv")

target = next((c for c in target_candidates if c in df.columns), None)

if target is None:
    print("No common target column found. Available columns:", df.columns.tolist())
else:
    X = df.drop(columns=[target])
    y = df[target]

    # keep only numeric features for the model
    X = X.select_dtypes(include="number")
    if X.shape[1] == 0:
        raise ValueError("No numeric features found after selecting X.")
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, shuffle=True
    )

    is_regression = y.dtype.kind in "fci" and y.nunique() > 10
    if is_regression:
        model = lgb.LGBMRegressor(random_state=42)
        eval_metric = "rmse"
    else:
        model = lgb.LGBMClassifier(random_state=42)
        eval_metric = "logloss"
    
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_test, y_test)],
        eval_metric=eval_metric,
        early_stopping_rounds=20,
        verbose=False,
    )

    y_pred = model.predict(X_test)

    if is_regression:
        print("RMSE:", mean_squared_error(y_test, y_pred, squared=False))
        print("R2:", r2_score(y_test, y_pred))
        plt.figure(figsize=(6, 5))
        plt.scatter(y_test, y_pred, alpha=0.4)
        plt.xlabel("Actual")
        plt.ylabel("Predicted")
        plt.title("LightGBM regression: actual vs predicted")
        plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color="red", linestyle="--")
        plt.show()
    else:
        print("Accuracy:", accuracy_score(y_test, y_pred))
        plt.figure(figsize=(6, 5))
        plt.hist([y_pred, y_test], label=["predicted", "actual"], bins=10, alpha=0.6)
        plt.legend()
        plt.title("LightGBM classification: predicted vs actual distribution")
        plt.show()

    ax = lgb.plot_importance(model, max_num_features=20, importance_type="gain")
    ax.set_title("LightGBM feature importance")
    plt.show()
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
import pandas as pd

import matplotlib.pyplot as plt

# choose a target column from common airline delay labels
target_candidates = ["ArrDelay", "Delay", "IsDelayed", "Cancelled", "FlightStatus"]
if 'df' not in globals():
    df = pd.read_csv("data/Airlines.csv")

target = next((c for c in target_candidates if c in df.columns), None)

if target is None:
    print("No common target column found. Available columns:", df.columns.tolist())
else:
    X = df.drop(columns=[target])
    y = df[target]

    # keep only numeric features for the model
    X = X.select_dtypes(include="number")
    if X.shape[1] == 0:
        raise ValueError("No numeric features found after selecting X.")
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, shuffle=True
    )

    is_regression = y.dtype.kind in "fci" and y.nunique() > 10
    if is_regression:
        model = lgb.LGBMRegressor(random_state=42)
        eval_metric = "rmse"
    else:
        model = lgb.LGBMClassifier(random_state=42)
        eval_metric = "logloss"
    
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_test, y_test)],
        eval_metric=eval_metric,
        early_stopping_rounds=20,
        verbose=False,
    )

    y_pred = model.predict(X_test)

    if is_regression:
        print("RMSE:", mean_squared_error(y_test, y_pred, squared=False))
        print("R2:", r2_score(y_test, y_pred))
        plt.figure(figsize=(6, 5))
        plt.scatter(y_test, y_pred, alpha=0.4)
        plt.xlabel("Actual")
        plt.ylabel("Predicted")
        plt.title("LightGBM regression: actual vs predicted")
        plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color="red", linestyle="--")
        plt.show()
    else:
        print("Accuracy:", accuracy_score(y_test, y_pred))
        plt.figure(figsize=(6, 5))
        plt.hist([y_pred, y_test], label=["predicted", "actual"], bins=10, alpha=0.6)
        plt.legend()
        plt.title("LightGBM classification: predicted vs actual distribution")
        plt.show()

    ax = lgb.plot_importance(model, max_num_features=20, importance_type="gain")
    ax.set_title("LightGBM feature importance")
    plt.show()

TypeError: LGBMClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'